# L4: Training the Model

## Changes from the tutorial

This notebook uses a modified `helper.py`; upload **both files** to Colab. The changes are:

1. **Correctness:** training examples now contain separate inputs, next-token targets, and a loss mask. Padding positions no longer lower the loss artificially.
2. **Correctness:** the dataset returns arrays shaped `[batch, sequence]`, removing the tutorial's list-pytree transpose workaround.
3. **Hardware adaptation:** the global batch is chosen from the detected CPU/GPU/TPU device count and sharded across all devices. Set `per_device_batch_size` below to override it.
4. **Tutorial fidelity:** the tutorial model architecture is unchanged. In particular, its `feed_forward_dim` argument remains unused; that is worth exploring later as a separate lesson rather than mixing it into this debugging pass.
5. **Memory/output:** unused logits are no longer returned from the training step, and logging occurs about 100 times per epoch.
6. **Checkpoint clarity:** retraining saves `model_checkpoint_v2.orbax`, keeping the masked-loss run separate from the earlier checkpoint.

The sampling fix is in `helper.py` and L5: temperature greater than zero uses seeded top-k categorical sampling; temperature zero remains greedy. See `TUTORIAL_CHANGES.md` for the full rationale.

In [ ]:
import jax
import jax.numpy as jnp
import optax
import tiktoken
from flax import nnx

In [ ]:
from helper import (
    MiniGPT,
    create_data_shardings,
    get_hardware_config,
    load_and_preprocess_hf_data,
)

In [ ]:
# Each training example contains at most 128 tokens. This is the model's
# context-window size, not a cap on the total number of tokens generated.
maxlen = 128
# Use the same fixed vocabulary for preprocessing, training, and generation.
tokenizer = tiktoken.get_encoding("gpt2")

# Leave this as None for hardware-aware defaults. If memory allows, increase
# it; if an out-of-memory error occurs, lower it and rerun from this cell.
per_device_batch_size = None
hardware = get_hardware_config(per_device_batch_size)
batch_sharding, replicated_sharding = create_data_shardings()
print(
    f"Backend: {hardware.platform} | devices: {hardware.device_count} | "
    f"per-device batch: {hardware.per_device_batch_size} | "
    f"global batch: {hardware.batch_size}"
)

In [ ]:
# text_dl, batches_per_epoch = load_and_preprocess_data(
#     file_path='TinyStories-1000.txt',
#     batch_size=32,
#     maxlen=128,
#     max_stories=100,
#     shuffle=False,
#     seed=42
# )

text_dl, batches_per_epoch = load_and_preprocess_hf_data(
      batch_size=hardware.batch_size,  # Global batch split across devices.
      maxlen=maxlen,      # Truncate/pad each training sequence to 128 tokens.
      split="train",      # Train on the TinyStories training split.
      max_stories=None,   # Stream every story; use an integer for a quicker run.
      num_epochs=1,       # One sampler pass per outer training-loop epoch.
      shuffle=True,       # Randomize story order within each pass.
      seed=42,            # Make that shuffled order reproducible.
  )

In [ ]:
model = MiniGPT(maxlen=maxlen)
# Keep one model replica per device; each replica receives a batch shard.
nnx.update(model, jax.device_put(nnx.state(model), replicated_sharding))

In [ ]:
def loss_fn(model, batch):
    logits = model(batch["inputs"])
    token_losses = optax.softmax_cross_entropy_with_integer_labels(
        logits, batch["targets"]
    )
    # Padding has mask 0, so only real next-token predictions affect training.
    loss_mask = batch["loss_mask"]
    return (token_losses * loss_mask).sum() / loss_mask.sum().clip(min=1)

In [ ]:
num_epochs = 3  # Make three complete passes over the training stories.
# The learning-rate schedule must span every optimizer update in all epochs.
total_steps = batches_per_epoch * num_epochs
# Ramp up gently for the first 10% of training to stabilize early updates.
warmup_steps = max(1, total_steps // 10)
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

In [ ]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,          # Start with no update at the first step.
    peak_value=3e-4,         # Keep the tutorial learning rate explicit.
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-5           # Retain a small learning rate at the end.
)

## Use an optimizer from optax

In [ ]:
optimizer = nnx.Optimizer(
    model,
    # AdamW adapts updates per parameter; 0.01 decay regularizes the weights.
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
    wrt=nnx.Param,
)

In [ ]:
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

## Define and JIT-compile the training step

In [ ]:
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_fn = nnx.value_and_grad(loss_fn)
    loss, grads = grad_fn(model, batch)

    metrics.update(loss=loss)
    optimizer.update(model, grads)

## Run the training loop

In [ ]:
%%time

metrics_history: dict[str, list[float]] = {'train_loss': []}

# Keep notebook output manageable while still showing about 100 updates/epoch.
log_every = max(1, batches_per_epoch // 100)

for epoch in range(num_epochs):
    for step, batch in enumerate(text_dl):
        # Grain emits [global_batch, sequence] arrays. Shard the leading axis
        # over every visible GPU/TPU core before the compiled training step.
        batch = jax.tree.map(
            lambda value: jax.device_put(jnp.asarray(value), batch_sharding),
            batch,
        )
        train_step(model, optimizer, metrics, batch)

        if (step + 1) % log_every == 0:
            for metric, value in metrics.compute().items():
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()

            # `step` restarts at zero each epoch; the schedule itself does not.
            global_step = epoch * batches_per_epoch + step
            current_lr = lr_schedule(global_step)
            print(f"\nEpoch: {epoch + 1}, Step {step + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")

# Wait for any asynchronously dispatched accelerator work to finish.
jax.block_until_ready(nnx.state(model))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(metrics_history['train_loss'])
plt.title(f'Training Loss — {num_epochs} epochs, full training split')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

### Training loss for 3 epochs, 2,000,000 stories

<img src="training_loss.png" width="570">


## Save model checkpoints

In [ ]:
from pathlib import Path

import orbax

# Keep the corrected masked-loss run separate from the tutorial checkpoint.
checkpoint_path = Path.cwd() / "model_checkpoint_v2.orbax"

checkpointer = orbax.checkpoint.PyTreeCheckpointer()

checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"Model saved as {checkpoint_path}")